In [73]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

In [74]:
df=pd.read_csv(r"C:\Users\USER\Desktop\5thsem_OJT\project1_diabeties\data\prosessed\fizan_P1W2_features.csv")

In [75]:
feature_cols = ["age","pregnancies","glucose","blood_pressure",
                "skin_thickness","insulin","bmi","diabetes_pedigree",
                "bmi_catogory","glucose_band"]

#create a list for wanted columns
#then save independent and depend variables

x = df[feature_cols].copy()
y = df["outcome"]
x

,age,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree,bmi_catogory,glucose_band
0,79.0,0,124.0,71.0,30.0,124.0,23.4,0.588,normal,pre_diebetic
1,37.0,0,153.0,85.0,24.0,42.0,33.9,0.192,obeese,diebetic
2,39.0,0,142.0,68.0,22.0,159.0,26.9,0.777,overweight,diebetic
3,68.0,7,121.0,88.0,26.0,124.0,29.0,1.217,overweight,pre_diebetic
4,75.0,0,107.0,84.0,31.0,101.0,21.2,0.278,normal,pre_diebetic
...,...,...,...,...,...,...,...,...,...,...
945,64.0,2,117.0,81.0,34.0,124.0,40.3,0.641,obeese,pre_diebetic
946,71.0,0,140.0,64.0,37.0,124.0,33.4,0.116,obeese,diebetic
947,40.0,0,130.0,73.0,26.0,258.0,20.1,0.709,normal,diebetic
948,52.0,4,123.0,76.0,22.0,189.0,38.9,0.540,obeese,pre_diebetic


In [76]:
x=pd.get_dummies(x,drop_first=True)
#drop_first is used for after new column add before columns removing function
#get_dummies is used for value to numerical value function
x.shape

(950, 13)

In [77]:
x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)

In [78]:
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)#Calculates the mean and standard deviation from x train
x_test_scaled=scaler.transform(x_test)#Uses those values to standardize x_train.

In [79]:
logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled,y_train)
knn = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)
decision_tree = DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train, y_train)

In [80]:
print("Logistic Regression", accuracy_score(y_test, logreg.predict(x_test_scaled)))

print("Decision Tree", accuracy_score(y_test, decision_tree.predict(x_test)))

print("KNN", accuracy_score(y_test, knn.predict(x_test_scaled)))

Logistic Regression 0.7526315789473684
Decision Tree 0.7210526315789474
KNN 0.7473684210526316


In [81]:
print("\nTest set:", len(y_test), "patients |",
      int(np.sum(y_test)), "of them diabetic")


Test set: 190 patients | 55 of them diabetic


In [82]:
models = [("Logistic Regression",logreg,x_test_scaled),
          ("decision_tree(d=3)",decision_tree, x_test),
          ("knn(k=25)",      knn, x_test_scaled)]

In [83]:
for name, m, Xt in models:
    print(f"{name:22s} accuracy = {accuracy_score(y_test, m.predict(Xt)):.4f}")

print("\nTest set:", len(y_test), "patients |", int(np.sum(y_test)), "of them diabetic")

Logistic Regression    accuracy = 0.7526
decision_tree(d=3)     accuracy = 0.7211
knn(k=25)              accuracy = 0.7474

Test set: 190 patients | 55 of them diabetic



buiI a contruction metrix for every modeI

In [84]:
rows = []
for name,m,Xt in models:
    cm = confusion_matrix(y_test,m.predict(Xt))
    tn, fp, fn, tp = cm.ravel()
    rows.append({"models": name, "correct_negative": int(tn),"False_alarms":int(fp),
                 "missed_patients":int(fn),"found_patients":int(tp),
                 "accuracy":accuracy_score(y_test,m.predict(Xt))})
    

In [85]:
metrices = pd.DataFrame(rows)
metrices

,models,correct_negative,False_alarms,missed_patients,found_patients,accuracy
0,Logistic Regression,127,8,39,16,0.752632
1,decision_tree(d=3),121,14,39,16,0.721053
2,knn(k=25),131,4,44,11,0.747368


### Decision Tree (max_depth = 3)

The Decision Tree achieved an accuracy of **72.11%** on 190 test patients.

- **121** patients were correctly identified as non-diabetic.
- **14** patients were incorrectly identified as diabetic.
- **39** diabetic patients were missed by the model.
- **16** diabetic patients were correctly detected.

Overall, the model made **137 correct predictions out of 190 patients**.

### Logistic Regression — Error Analysis

- **False Positive (False Alarm):** 8
- **False Negative (Missed Patient):** 39

The model has **31 more False Negatives than False Positives**.

False Negatives are much higher and therefore **dominate the errors**.
The model misses more diabetic patients than it incorrectly identifies non-diabetic patients as diabetic.

score the modeI that Iast says yes

In [89]:
from  sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(x_train,y_train)
y_pred=dummy.predict(x_test)
confusion_matrix(y_test,y_pred)

array([[135,   0],
       [ 55,   0]])

In [90]:
accuracy_score(y_test,y_pred)

0.7105263157894737

In [91]:
y_test.value_counts(normalize=True)

outcome
0    0.710526
1    0.289474
Name: proportion, dtype: float64